# ===============================================================
# Phase 4: Retrieval Quality Test (Chroma)
# ===============================================================


In [1]:

from pathlib import Path
from dotenv import load_dotenv
import os, json
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

load_dotenv()

# ตั้ง root โปรเจกต์แบบชัด ๆ (แก้ได้ถ้าพาธไม่ตรงเครื่องคุณ)
PROJECT_ROOT = Path(r"D:\mini-jane-demo")

# ค่าจาก .env (ถ้าไม่มีจะ fallback)
CHROMA_DIR  = Path(os.getenv("CHROMA_PERSIST_DIR", PROJECT_ROOT / "vectorstore"))
EMBED_MODEL = os.getenv("EMBED_MODEL", "BAAI/bge-m3")
COLLECTION_NAME = "jenosize-ideas"

# connect Chroma + collection
client = chromadb.PersistentClient(path=str(CHROMA_DIR), settings=Settings(allow_reset=True))
collection = client.get_or_create_collection(name=COLLECTION_NAME)

# embedder (ใช้ตัวเดียวกับ 03)
embedder = SentenceTransformer(EMBED_MODEL)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("CHROMA_DIR   :", CHROMA_DIR)
print("COLLECTION   :", COLLECTION_NAME)
print("COUNT        :", collection.count())
print("EMBED_MODEL  :", EMBED_MODEL)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Exception ignored in: <function tqdm.__del__ at 0x00000280462BEFC0>
Traceback (most recent call last):
  File "d:\mini-jane-demo\.venv\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "d:\mini-jane-demo\.venv\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm' object has no attribute 'disp'


PROJECT_ROOT : D:\mini-jane-demo
CHROMA_DIR   : vectorstore
COLLECTION   : jenosize-ideas
COUNT        : 446
EMBED_MODEL  : BAAI/bge-m3


In [2]:
def search(query, k=5, where=None, where_document=None, include_embeddings=False):
    q_emb = embedder.encode([query], normalize_embeddings=True).tolist()[0]
    include = ["documents", "metadatas", "distances"]
    if include_embeddings:
        include.append("embeddings")
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=include,
        where=where,
        where_document=where_document
    )
    return q_emb, res

def show_results(query, res, max_chars=200):
    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    dists = res.get("distances", [[]])[0]
    if not docs:
        print(f"❌ No results for: {query}")
        return
    print(f"\n🔎 Query: {query}  (top {len(docs)})")
    for i, (doc, m, d) in enumerate(zip(docs, metas, dists), 1):
        title = m.get("title","(no title)")
        cat   = m.get("category","-")
        sec   = m.get("section","-")
        url   = m.get("url","-")
        idx   = m.get("chunk_index","-")
        print(f"\n[{i}] dist={d:.4f} | {title}  [{cat}]  (#{idx} @ {sec})")
        print(url)
        print(doc[:max_chars].replace("\n"," ") + (" ..." if len(doc) > max_chars else ""))


In [3]:
# ไทย
q = "แนวโน้ม AI สำหรับธุรกิจสมัยใหม่"
q_emb, res = search(q, k=5)
show_results(q, res)

# อังกฤษ
q2 = "Benefits of Agentic AI in organizations"
q_emb2, res2 = search(q2, k=5)
show_results(q2, res2)



🔎 Query: แนวโน้ม AI สำหรับธุรกิจสมัยใหม่  (top 5)

[1] dist=0.7369 | 8 Trending Startup Businesses for Modern Entrepreneurs  [Experience the New World]  (#5 @ Body)
https://www.jenosize.com/en/ideas/experience-the-new-world/top-startup-trends
7. AI-Aided Engineering Tools The engineering tools for designing physical systems, from CAD/CAM to CFD software, have seen little advancement in decades. These tools often require complex simulations ...

[2] dist=0.7786 | Understanding 7 Megatrends for 2024 to Shape Your Tomorrow  [Experience the New World]  (#4 @ Body)
https://www.jenosize.com/en/ideas/experience-the-new-world/7-megatrends-in-2024
For example, leading cosmetics and skincare companies are reformulating products to be organic and cruelty-free to meet the preferences of modern consumers, enabling organizations to sustainably grow. ...

[3] dist=0.7956 | 12 Cloud Computing Trends Shaping the Future of Technology  [Futurist]  (#3 @ Body)
https://www.jenosize.com/en/ideas/futurist/c

In [4]:
q = "AI trends for businesses"
q_emb, res = search(q, k=5, where={"category": "Futurist"})
show_results(q, res)



🔎 Query: AI trends for businesses  (top 5)

[1] dist=0.7009 | 12 Cloud Computing Trends Shaping the Future of Technology  [Futurist]  (#3 @ Body)
https://www.jenosize.com/en/ideas/futurist/cloud-computing-trends
Emerging cloud computing trends in security include zero trust architecture, which reduces risks by requiring strict authentication for every access, and confidential computing, which protects data du ...

[2] dist=0.7502 | 12 Cloud Computing Trends Shaping the Future of Technology  [Futurist]  (#2 @ Body)
https://www.jenosize.com/en/ideas/futurist/cloud-computing-trends
Additionally, Multi-cloud helps businesses comply with regional regulations, especially for those managing cross-border data and adhering to legal requirements in different countries, such as internat ...

[3] dist=0.7734 | 16 AI Customer Service Platforms to Elevate Customer Support  [Futurist]  (#5 @ Body)
https://www.jenosize.com/en/ideas/futurist/ai-customer-service-platform
This reduces the workload of in

In [5]:
q = "effective strategies"
q_emb, res = search(q, k=5, where_document={"$contains": "marketing"})
show_results(q, res)



🔎 Query: effective strategies  (top 5)

[1] dist=0.9197 | 5 Types of Brand Loyalty and Tips to Strengthen Each One  [Real-time Marketing]  (#4 @ Body)
https://www.jenosize.com/en/ideas/real-time-marketing/brand-loyalty-pyramid
Recommended Strategies - Use content marketing to share meaningful brand stories through storytelling or brand purpose campaigns. - Build a community where customers can engage—such as online groups o ...

[2] dist=0.9197 | 5 Types of Brand Loyalty and Tips to Strengthen Each One  [Real-time Marketing]  (#4 @ Body)
https://www.jenosize.com/ideas/real-time-marketing/brand-loyalty-pyramid
Recommended Strategies - Use content marketing to share meaningful brand stories through storytelling or brand purpose campaigns. - Build a community where customers can engage—such as online groups o ...

[3] dist=0.9222 | What is Reciprocity Marketing? A Strategy of Giving Back  [Understand People & Consumer]  (#4 @ Body)
https://www.jenosize.com/ideas/understand-people-and-con

In [6]:
for k in [3, 5, 8]:
    q = "customer experience personalization"
    _, res = search(q, k=k)
    print(f"\n=== k={k} ===")
    show_results(q, res, max_chars=140)



=== k=3 ===

🔎 Query: customer experience personalization  (top 3)

[1] dist=0.7293 | Seamless O2O Marketing and Customer Experience Strategies  [Experience the New World]  (#4 @ Body)
https://www.jenosize.com/en/ideas/experience-the-new-world/customer-experience-for-o2o-marketing
- Provide convenience for customers to initiate purchases from one channel and seamlessly transition to another. For example, customers can  ...

[2] dist=0.7416 | What is Reciprocity Marketing? A Strategy of Giving Back  [Understand People & Consumer]  (#3 @ Body)
https://www.jenosize.com/ideas/understand-people-and-consumer/reciprocity-marketing
This value can take various forms, such as: - Creating guides, infographics, or skill-based instructional videos free of charge. - Building  ...

[3] dist=0.7710 | How to Create a Loyalty Program For Repeat Purchases  [Transformation & Technology]  (#4 @ Body)
https://www.jenosize.com/en/ideas/transformation-and-technology/loyalty-program-tips
Use Customer Behavior

In [7]:
import numpy as np

def cosine_sim(a, b):
    a = np.asarray(a); b = np.asarray(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

def mmr(query_emb, doc_embs, docs, metas, k=5, lambda_mult=0.5):
    """
    query_emb: 1D vec
    doc_embs : list of 1D vec
    docs     : list of texts
    metas    : list of metadatas
    k        : top results to return
    lambda_mult: trade-off (1.0 = relevance only, 0.0 = diversity only)
    """
    n = len(doc_embs)
    if n == 0:
        return []

    # relevance scores (similarity to query)
    rel = np.array([cosine_sim(query_emb, e) for e in doc_embs])

    selected = []
    candidates = list(range(n))
    selected_scores = []

    # pick the best by relevance first
    first = int(np.argmax(rel))
    selected.append(first); candidates.remove(first)
    selected_scores.append(rel[first])

    while len(selected) < min(k, n) and candidates:
        mmr_scores = []
        for c in candidates:
            # diversity = max similarity to any selected doc
            div = max(cosine_sim(doc_embs[c], doc_embs[s]) for s in selected)
            score = lambda_mult * rel[c] - (1 - lambda_mult) * div
            mmr_scores.append((score, c))
        mmr_scores.sort(reverse=True)
        best = mmr_scores[0][1]
        selected.append(best); candidates.remove(best); selected_scores.append(rel[best])

    return [(docs[i], metas[i], selected_scores[idx]) for idx,i in enumerate(selected)]

# ใช้งาน: ดึงมาเยอะหน่อย แล้วทำ MMR เลือก k=5
query = "real-time marketing tactics"
q_emb, res = search(query, k=20, include_embeddings=True)  # include embeddings
docs = res["documents"][0]
metas = res["metadatas"][0]
embs  = res["embeddings"][0]

reranked = mmr(q_emb, embs, docs, metas, k=5, lambda_mult=0.6)

print(f"\n🔎 Query (MMR): {query}")
for i, (doc, m, rel_score) in enumerate(reranked, 1):
    print(f"\n[{i}] rel≈{rel_score:.4f} | {m.get('title','(no title)')} [{m.get('category','-')}] #{m.get('chunk_index','-')}")
    print(m.get('url','-'))
    print(doc[:200].replace("\n"," ") + (" ..." if len(doc) > 200 else ""))



🔎 Query (MMR): real-time marketing tactics

[1] rel≈0.6024 | LIVE Commerce is Booming! 7 Tips to Boost Sales via LIVE [Real-time Marketing] #6
https://www.jenosize.com/en/ideas/real-time-marketing/live-commerce-techniques-to-boost-sales
Special Tip: Review comments and feedback from your viewers to understand what they liked or didn’t like. Use this information to enhance your strategy and tailor your LIVE commerce sessions to better ...

[2] rel≈0.5713 | Technostalgia: Memory-Driven Marketing Built on Technology [Transformation & Technology] #2
https://www.jenosize.com/en/ideas/transformation-and-technology/technostalgia-marketing
These are examples of nostalgia marketing that go beyond product features—they sell a feeling and foster emotional ties using elements of the past as a bridge. Why Is Technostalgia Trending in Retro M ...

[3] rel≈0.5766 | What Is Micro-Moment Marketing? Win Customers Instantly [Real-time Marketing] #4
https://www.jenosize.com/ideas/real-time-marketing/micr

In [ ]:
REPORT = PROJECT_ROOT / "data" / "processed" / "retrieval_eval.txt"

cases = [
    {"q":"AI trends for 2030", "expect_cat": "Futurist", "contains":["AI","trend"]},
    {"q":"customer service platforms", "expect_cat": "Real-time Marketing", "contains":["customer","service"]},
    {"q":"sustainability in business", "expect_cat": "Utility for Our World", "contains":["sustain","green"]},
]

lines = []
ok = 0
for case in cases:
    q = case["q"]
    _, res = search(q, k=5)
    docs = res["documents"][0]
    metas = res["metadatas"][0]
    hit = False
    for doc, m in zip(docs, metas):
        if m.get("category") == case["expect_cat"]:
            if any(tok.lower() in doc.lower() for tok in case["contains"]):
                hit = True; break
    lines.append(f"{'✅' if hit else '❌'}  {q}  -> expect_cat={case['expect_cat']}")
    ok += int(hit)

REPORT.write_text("\n".join(lines) + f"\n\nScore: {ok}/{len(cases)}", encoding="utf-8")
print("📝 Wrote:", REPORT)
print("\n".join(lines))


📝 Wrote: D:\mini-jane-demo\data\processed\retrieval_eval.txt
✅  AI trends for 2030  -> expect_cat=Futurist
❌  customer service platforms  -> expect_cat=Real-time Marketing
✅  sustainability in business  -> expect_cat=Utility for Our World


In [16]:
cases = [
    {"q":"AI trends for 2030", "expect_any":["Futurist"]},
    {"q":"customer service platforms", "expect_any":["Understand People & Consumer"]},
    {"q":"sustainability in business", "expect_any":["Utility for Our World"]},
]

ok = 0
for case in cases:
    _, res = search(case["q"], k=5)
    metas = res["metadatas"][0] if res.get("metadatas") else []
    hit = any(m.get("category") in case["expect_any"] for m in metas)
    print(f"{'✅' if hit else '❌'} {case['q']} -> expect_any={case['expect_any']}")
    ok += int(hit)
print(f"\nScore: {ok}/{len(cases)}")


✅ AI trends for 2030 -> expect_any=['Futurist']
❌ customer service platforms -> expect_any=['Understand People & Consumer']
✅ sustainability in business -> expect_any=['Utility for Our World']

Score: 2/3


In [9]:
q = "customer service platforms"
_, res = search(q, k=8)  # ใช้ฟังก์ชัน search เดิมของเล่มนี้
show_results(q, res, max_chars=160)



🔎 Query: customer service platforms  (top 8)

[1] dist=0.6758 | 16 AI Customer Service Platforms to Elevate Customer Support  [Futurist]  (#5 @ Body)
https://www.jenosize.com/en/ideas/futurist/ai-customer-service-platform
This reduces the workload of internal support teams and is highly suitable for enterprises with large workforces seeking workflow automation. Try ServiceNow htt ...

[2] dist=0.7101 | 16 AI Customer Service Platforms to Elevate Customer Support  [Futurist]  (#4 @ Body)
https://www.jenosize.com/en/ideas/futurist/ai-customer-service-platform
It is ideal for industries with high customer volume, such as telecom, finance, and retail. Try LivePerson https://www.liveperson.com/request-demo 7. Ada Ada is ...

[3] dist=0.7114 | 16 AI Customer Service Platforms to Elevate Customer Support  [Futurist]  (#7 @ Body)
https://www.jenosize.com/en/ideas/futurist/ai-customer-service-platform
Real-time responses reduce customer wait time An AI customer service platform can respond ins

In [13]:
import re, numpy as np

# 1) คำพ้อง/คีย์เวิร์ดที่ช่วยระบุ intent
KEYWORDS = {
    "customer_service": ["customer", "customer support", "cx", "helpdesk", "chatbot", "platform", "ticketing"]
}

def cosine_sim(a, b):
    a = np.asarray(a); b = np.asarray(b)
    return float(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-12))

def boosted_search(query, k=5, fetch_k=30, category_hint=None, keyword_group=None,
                   lambda_mmr=0.6, w_keyword=0.2, w_cat=0.15):
    """
    fetch_k: ดึงมาเยอะหน่อยก่อน rerank (เช่น 30)
    category_hint: ถ้าต้องการ boost หมวด เช่น "Real-time Marketing"
    keyword_group: ใช้ KEYWORDS[...] เพื่อเช็กคีย์เวิร์ด
    w_keyword, w_cat: น้ำหนักการบูสต์ (0.0–1.0)
    """
    # ขั้น 1: ดึงข้อมูลดิบ + embeddings สำหรับ rerank
    q_emb = embedder.encode([query], normalize_embeddings=True).tolist()[0]
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=fetch_k,
        include=["documents", "metadatas", "distances", "embeddings"]
    )
    docs  = res["documents"][0]
    metas = res["metadatas"][0]
    embs  = res["embeddings"][0]

    if not docs:
        return []

    # ขั้น 2: relevance เบื้องต้น = cosine_sim กับ query
    rel = np.array([cosine_sim(q_emb, e) for e in embs])

    # ขั้น 3: keyword boost
    kw_boost = np.zeros(len(docs))
    if keyword_group and keyword_group in KEYWORDS:
        pats = [re.compile(re.escape(kw), re.I) for kw in KEYWORDS[keyword_group]]
        for i, d in enumerate(docs):
            if any(p.search(d) for p in pats):
                kw_boost[i] = 1.0  # hit

    # ขั้น 4: category boost
    cat_boost = np.zeros(len(docs))
    if category_hint:
        for i, m in enumerate(metas):
            if m.get("category") == category_hint:
                cat_boost[i] = 1.0

    # ขั้น 5: คะแนนรวม (normalize rel -> 0..1 แล้วรวม boost)
    rel_n = (rel - rel.min()) / (rel.max() - rel.min() + 1e-9)
    score = rel_n + w_keyword*kw_boost + w_cat*cat_boost

    # ขั้น 6: MMR เพื่อความหลากหลายจากผู้ถูกคัด
    selected, cand = [], list(range(len(docs)))
    # เริ่มจากคนที่ได้คะแนนรวมสูงสุด
    first = int(np.argmax(score)); selected.append(first); cand.remove(first)
    while len(selected) < min(k, len(docs)) and cand:
        best, best_val = None, -1e9
        for c in cand:
            div = max(cosine_sim(embs[c], embs[s]) for s in selected)
            val = lambda_mmr*score[c] - (1-lambda_mmr)*div
            if val > best_val:
                best_val, best = val, c
        selected.append(best); cand.remove(best)

    out = []
    for i in selected:
        out.append((docs[i], metas[i], float(score[i])))
    return out

def show_boosted(query, items, max_chars=200):
    print(f"\n🔎 Boosted Query: {query}  (top {len(items)})")
    for idx, (doc, m, sc) in enumerate(items, 1):
        print(f"\n[{idx}] score≈{sc:.3f} | {m.get('title','(no title)')} [{m.get('category','-')}] #{m.get('chunk_index','-')}")
        print(m.get('url','-'))
        print(doc[:max_chars].replace("\n"," ") + (" ..." if len(doc) > max_chars else ""))


In [14]:
query = "customer service platforms"
items = boosted_search(
    query, 
    k=5, 
    fetch_k=30, 
    category_hint="Real-time Marketing",         # บอกหมวดที่อยากเน้น
    keyword_group="customer_service",            # กลุ่มคีย์เวิร์ด
    lambda_mmr=0.6, w_keyword=0.25, w_cat=0.2    # ปรับได้
)
show_boosted(query, items, max_chars=180)



🔎 Boosted Query: customer service platforms  (top 5)

[1] score≈1.250 | 16 AI Customer Service Platforms to Elevate Customer Support [Futurist] #5
https://www.jenosize.com/en/ideas/futurist/ai-customer-service-platform
This reduces the workload of internal support teams and is highly suitable for enterprises with large workforces seeking workflow automation. Try ServiceNow https://www.servicenow. ...

[2] score≈1.095 | 16 AI Customer Service Platforms to Elevate Customer Support [Futurist] #7
https://www.jenosize.com/en/ideas/futurist/ai-customer-service-platform
Real-time responses reduce customer wait time An AI customer service platform can respond instantly, 24/7, across websites, social media, or mobile apps. This reduces customer frus ...

[3] score≈1.101 | 16 AI Customer Service Platforms to Elevate Customer Support [Futurist] #4
https://www.jenosize.com/en/ideas/futurist/ai-customer-service-platform
It is ideal for industries with high customer volume, such as telecom, finance

In [15]:
cases = [
    {"q":"AI trends for 2030", "expect_cat":"Futurist", "kw":None, "cat_hint":None},
    {"q":"customer service platforms", "expect_cat":"Real-time Marketing", "kw":"customer_service", "cat_hint":"Real-time Marketing"},
    {"q":"sustainability in business", "expect_cat":"Utility for Our World", "kw":None, "cat_hint":"Utility for Our World"},
]

ok = 0
for c in cases:
    items = boosted_search(
        c["q"], k=5, fetch_k=30,
        category_hint=c["cat_hint"], keyword_group=c["kw"],
        lambda_mmr=0.6, w_keyword=0.25, w_cat=0.2
    )
    hit = any(m.get("category")==c["expect_cat"] for _,m,_ in items)
    print(f"{'✅' if hit else '❌'}  {c['q']}  -> expect_cat={c['expect_cat']}")
    ok += int(hit)

print(f"\nScore (boosted): {ok}/{len(cases)}")


✅  AI trends for 2030  -> expect_cat=Futurist
❌  customer service platforms  -> expect_cat=Real-time Marketing
✅  sustainability in business  -> expect_cat=Utility for Our World

Score (boosted): 2/3
